## tl;dr

V11.18 条件风险配置 / Conditional risk allocation.
This notebook verifies saved evidence; it does not start a new experiment or certify alpha.


## Context & Methods

44 accounts, four primary identities, CNY3m, 82/164bps; 2022 training and reused 2023–2024 development. No 2025/26 data access.
### Key Assumptions

Conditional risk allocation is not a new stock selector. Gross basket proxy labels differ from executable portfolio returns. All 44 outcomes remain visible.
Required cell dependencies: Python3.10+ and DuckDB (`pip install -e '.[research]'`).


## Data

### 1. Verify bounded evidence and immutable result identity


In [1]:
import json, hashlib, math
from pathlib import Path
import duckdb
repo = Path.cwd()
if not (repo / 'docs/V11_18_RESULT.summary.json').exists():
    repo = repo.parent
operation = repo / 'artifacts/conditional-risk/epoch-001'
summary = json.loads((repo / 'docs/V11_18_RESULT.summary.json').read_text(encoding='utf-8'))
audit = json.loads((operation / 'INDEPENDENT_AUDIT.json').read_text(encoding='utf-8'))
assert audit['pass'] and summary['independent_audit_pass']
assert hashlib.sha256((operation/'RESULT.json').read_bytes()).hexdigest() == summary['source_result_sha256']
assert len(summary['rows']) == 44 and summary['raw_trial_lower_bound'] == 3600
print({'accounts':44, 'states':summary['states'], 'labels':summary['proxy_labels'],
       'coverage':summary['coverage'], 'raw_trial_lower_bound':3600})


{'accounts': 44, 'states': 726, 'labels': 95, 'coverage': {'2023': 1.0, '2024': 1.0}, 'raw_trial_lower_bound': 3600}


### 2. Verify the source-query identity and fit horizons

This binds the query already executed by the independent audit; the notebook does not rerun the raw-source join.


In [2]:
query_source = (repo/'scripts/conditional_risk_source_audit.sql').read_text(encoding='utf-8')
assert query_source == audit['source_query']
for identity, model in summary['models_summary'].items():
    assert model['maximum_label_end'] <= model['fit_cutoff']
    assert model['training_signal_dates'] >= 30
    print(identity, model)


direct-2023 {'fit_cutoff': '2022-12-23', 'maximum_label_end': '2022-12-23', 'mean_return': -0.0015080497977713813, 'mean_second': 0.0006392695256693367, 'rotation': 0, 'training_mean_exposure': {'mean': 0.3423913043478261, 'mean_second': 0.3423913043478261, 'risk_only': 0.25}, 'training_signal_dates': 46}
direct-2024 {'fit_cutoff': '2023-12-22', 'maximum_label_end': '2023-12-20', 'mean_return': 0.00023930176371939285, 'mean_second': 0.0004702216814528631, 'rotation': 0, 'training_mean_exposure': {'mean': 0.3271276595744681, 'mean_second': 0.31648936170212766, 'risk_only': 0.25}, 'training_signal_dates': 94}
shuffle-2023 {'fit_cutoff': '2022-12-23', 'maximum_label_end': '2022-12-23', 'mean_return': -0.0015080497977713817, 'mean_second': 0.0006392695256693367, 'rotation': 15, 'training_mean_exposure': {'mean': 0.33152173913043476, 'mean_second': 0.34782608695652173, 'risk_only': 0.25}, 'training_signal_dates': 46}
shuffle-2024 {'fit_cutoff': '2023-12-22', 'maximum_label_end': '2023-12-20

## Results

### 3. Independently recompute all 44 saved account aggregates


In [3]:
query = (repo/'scripts/lead_challenge_audit.sql').read_text(encoding='utf-8')
query = query.replace('__ACCOUNT_GLOB__', (operation/'accounts/*.jsonl').as_posix())
with duckdb.connect() as con:
    cursor = con.execute(query)
    fields = [x[0] for x in cursor.description]
    actual = {r[0]:dict(zip(fields,r)) for r in cursor.fetchall()}
assert len(actual) == 44
for row in summary['rows']:
    for field in ('net_return','final_nav','cost_cny','max_drawdown'):
        assert math.isclose(actual[row['account_key']][field], row[field], rel_tol=1e-10, abs_tol=1e-6)
print({'independently_recomputed_accounts':44,'aggregate_reconciliation':True})


{'independently_recomputed_accounts': 44, 'aggregate_reconciliation': True}


### 4. Inspect all primary identities and both costs


In [4]:
for r in summary['primary_comparisons']:
    print({k:r[k] for k in ('identity','roundtrip_bps','return2023','return2024','net_return',
                          'sharpe','max_drawdown','minimum_control_increment','mean_cash_fraction')})
print({'exploratory_survivors':summary['screen_survived'],'validated_alpha':summary['validated_alpha']})


{'identity': 'lowvol-mean', 'roundtrip_bps': 164, 'return2023': -0.017464814364993364, 'return2024': 0.06625208285338502, 'net_return': 0.04763018816006581, 'sharpe': 0.3543338325088374, 'max_drawdown': -0.0801717225493811, 'minimum_control_increment': 0.008084837262765765, 'mean_cash_fraction': 0.6089599859108931}
{'identity': 'lowvol-mean', 'roundtrip_bps': 82, 'return2023': 5.6379690400287785e-05, 'return2024': 0.12259621564092349, 'net_return': 0.12265950726800035, 'sharpe': 0.8336497548713632, 'max_drawdown': -0.06496614028464409, 'minimum_control_increment': -0.07904034682716743, 'mean_cash_fraction': 0.60926860015968}
{'identity': 'lowvol-mean_second', 'roundtrip_bps': 164, 'return2023': -0.017464814364993364, 'return2024': 0.08003378833339414, 'net_return': 0.06117119871223253, 'sharpe': 0.39244983385163285, 'max_drawdown': -0.08209990564734015, 'minimum_control_increment': 0.021625847814932486, 'mean_cash_fraction': 0.6218250768306686}
{'identity': 'lowvol-mean_second', 'round

## Takeaways

Only complete screen survivors may be frozen for deeper registered challenges. Reused development evidence does not establish independent alpha. Cash/risk reduction is not alpha.
All plain-Python code cells are executed in order by scripts/build_conditional_risk_notebook.py. nbformat, nbclient and ipykernel are absent on this host; Jupyter kernel/frontend and pixel QA were not run. Optional after installation: `python -m jupyter nbconvert --execute --to notebook --inplace notebooks/V11_18_AUDIT.ipynb`.
